# Spherinator & HiPSter: an interactive sky of emoji

*Author: Bernd Doser (bernd.doser@h-its.org) &middot; Date: 2026-09-11 &middot; License: [Apache-2.0](../LICENSE)*

This notebook is part 2 of 2. It turns a trained
[Spherinator](https://github.com/HITS-AIN/Spherinator) model of the
[Emoji Dataset](https://huggingface.co/datasets/valhalla/emoji-dataset) into an
explorable sky with [HiPSter](https://github.com/HITS-AIN/HiPSter) and
[Aladin Lite](https://aladin.cds.unistra.fr/AladinLite/). It is pure inference —
no GPU and no training loop — so instead of training a model itself it
downloads a fully trained one, published on Hugging Face at
[`bernddoser/emoji`](https://huggingface.co/bernddoser/emoji):

1. **Download** the trained encoder ONNX graph from Hugging Face.
2. **Encode** every real emoji with it into a catalogue of positions on the
   latent sphere.
3. **Project** the real dataset onto a [HiPS](https://www.ivoa.net/documents/HiPS/)
   tiling with HiPSter: every HEALPix cell shows the real emoji whose encoded
   position lands closest to that cell's centre.
4. **Explore** the result in Aladin Lite through the `ipyaladin` widget, with
   the point catalogue overlaid on the tiling.

The key idea is that a *spherical* latent space is directly a sky: once every
emoji has a position on $S^2$, astronomical sky-viewers become general-purpose
tools for exploring a learned representation.

The companion notebook, [`emojis_training.ipynb`](emojis_training.ipynb),
trains a small demo version of this same kind of model from scratch in well
under two minutes — worth reading first if you want to see where the
production model downloaded below comes from.

In [ ]:
# HiPSter and ipyaladin for the tiling/viewer, huggingface_hub to fetch the
# trained model, spherinator to load the dataset for the emoji catalogue.
# Installed by `uv sync` from pyproject.toml; the fallback keeps the notebook
# runnable in a bare Jupyter container (see compose.yml).
try:
    import hipster
    import huggingface_hub
    import ipyaladin
    import spherinator
except ImportError:
    %pip -q install git+https://github.com/HITS-AIN/Spherinator git+https://github.com/HITS-AIN/HiPSter ipyaladin huggingface_hub
    import hipster
    import huggingface_hub
    import ipyaladin
    import spherinator

print(f"spherinator     {spherinator.__version__}")
print(f"hipster         {hipster.__version__}")
print(f"ipyaladin       {ipyaladin.__version__}")
print(f"huggingface_hub {huggingface_hub.__version__}")

In [ ]:
import os

import numpy as np

# Every artefact this notebook produces lands under `output/`, which the HTTP
# server further down exposes to Aladin Lite at BASE_URL.
OUTPUT_PATH = "output"
PORT = 8083
BASE_URL = f"http://localhost:{PORT}"

## 1. Download the trained model

Rather than training here, this notebook fetches a finished
[ONNX](https://onnx.ai/) graph from the Hugging Face model repo
[`bernddoser/emoji`](https://huggingface.co/bernddoser/emoji): a
`HuggingFaceResNetEncoder` mapped onto the same $S^2$ (`z_dim=3`) latent
sphere as the demo in `emojis_training.ipynb`, but trained on 128 × 128 emoji
for far longer than a two-minute demo allows. `hf_hub_download` caches the
file locally, so re-running this cell after the first download is instant.

`encoder.onnx` maps an image `x` `[N, 3, 128, 128]` to a direction
`[N, 3]` on the sphere — used below to place every real emoji on the sky.

The input/output names match what HiPSter's `Inference` class expects, exactly
as in the training notebook's own export.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO = "bernddoser/emoji"
HF_SUBDIR = "vae/resnet18/S2"

encoder_path = hf_hub_download(repo_id=HF_REPO, filename=f"{HF_SUBDIR}/encoder.onnx")
print(encoder_path)

## 2. Set up the HiPS page

`HTMLGenerator` writes a standalone `index.html` that loads a
[HiPS](https://www.ivoa.net/documents/HiPS/) tiling in Aladin Lite. Every task
below `register`s itself with it, so the page knows which layers exist —
the dataset projection generated further down, and the point catalogue of
every real emoji's encoded position. Running the equivalent from the shell is
`hipster --config <config.yaml>`; the YAML mirrors these arguments one-to-one
(see [`examples/`](https://github.com/HITS-AIN/HiPSter/tree/main/examples)).

In [ ]:
# `url` is where the browser will reach these files - see the HTTP server below.
html_generator = hipster.HTMLGenerator(
    root_path=OUTPUT_PATH,
    url=BASE_URL,
    title="Spherinator projection of the Emoji Dataset",
    aladin_lite_version="3.6.5",
)

## 3. A catalogue of the real emoji

To place the real data on the sky, we run the ONNX encoder over the whole
dataset and convert each latent direction to sky coordinates.

The conversion is `healpy.vec2ang`, which turns a unit vector into HEALPix
spherical coordinates $(\theta, \phi)$: colatitude $\theta$ measured from the
north pole, longitude $\phi$. Right ascension is $\phi$, and declination is
$90° - \theta$.

This uses a Spherinator `DataModule`, resizing every emoji to 128 × 128
to match the downloaded encoder's input size, and carries the `text` column —
the emoji's name — so clicking a source in Aladin identifies which emoji it is.
`validation_size=0.0` and `shuffle=False` keep the full dataset in its original
order. HiPSter ships `hipster.VOTableGenerator` for this step when reading
local parquet files with a flat image column; here we build the table directly
from the Hugging Face dataset.

In [ ]:
import torch
import torchvision.transforms.functional as TF


def resize_image(image: torch.Tensor) -> torch.Tensor:
    return TF.resize(image, [128, 128], antialias=True)


catalog_datamodule = spherinator.data.DataModule(
    path="valhalla/emoji-dataset",
    columns=[{"name": "image", "transform": resize_image}, "text"],
    return_dict=True,  # we need the metadata alongside the images
    validation_size=0.0,  # no split: encode every emoji
    test_size=0.0,
    batch_size=256,
    shuffle=False,
    num_workers=4,
)
catalog_datamodule.setup("fit")

encoder_inference = hipster.Inference(
    model_path=encoder_path,
    input_name="x",
)

latent, metadata = [], {"text": []}
for batch in catalog_datamodule.train_dataloader():
    latent.append(encoder_inference(batch["image"].numpy()))
    metadata["text"].extend(batch["text"])
latent = np.concatenate(latent)
print(f"encoded {len(latent)} emoji")

In [ ]:
import healpy
from astropy.table import Table

theta, phi = healpy.vec2ang(latent)  # colatitude and longitude, in radians

catalog = Table(
    {
        "ra": np.degrees(phi),
        "dec": 90.0 - np.degrees(theta),
        "text": metadata["text"],
    }
)
catalog["ra"].unit = "deg"
catalog["dec"].unit = "deg"
# UCDs let any VO client recognise these two columns as the sky position.
catalog["ra"].meta["ucd"] = "pos.eq.ra;meta.main"
catalog["dec"].meta["ucd"] = "pos.eq.dec;meta.main"

catalog.write(
    os.path.join(OUTPUT_PATH, "catalog.vot"), format="votable", overwrite=True
)

# Register the catalogue as a layer too, and re-render the standalone page so it
# offers both the dataset projection and the real emoji.
html_generator.add_votable(
    html_generator.VOTable(
        url=f"{BASE_URL}/catalog.vot",
        name="Emoji dataset",
        color="#ff3b30",
        shape="circle",
        size=8,
    )
)
html_generator.generate()

catalog[:5]

## 4. Project the real dataset onto a HiPS tiling

`hipster.DatasetProjection` builds a [HiPS](https://www.ivoa.net/documents/HiPS/)
survey straight from the real data: for every HEALPix cell it takes the real
emoji whose encoded direction lands closest to the cell centre and uses *that
actual image* as the tile — a nearest-neighbour projection of the dataset onto
the sphere, so panning across the sky pans across the latent space the
encoder actually learned.

`DatasetProjection` encodes the dataset itself, reading images straight from a
local Parquet file rather than through the `DataModule` used above — but it
reuses `encoder_inference`, so the ONNX session is only loaded once. The Emoji
Dataset on Hugging Face already ships as a single Parquet file, so
`snapshot_download` fetches just that.

In [ ]:
from huggingface_hub import snapshot_download

DATASET_REPO = "valhalla/emoji-dataset"

# Only the Parquet shard is needed, not the full dataset repo.
dataset_snapshot = snapshot_download(
    repo_id=DATASET_REPO, repo_type="dataset", allow_patterns="data/*.parquet"
)
dataset_directory = os.path.join(dataset_snapshot, "data")

dataset_projection = hipster.DatasetProjection(
    encoder=encoder_inference,
    image_maker=hipster.ImagePlotter(),
    data_directory=dataset_directory,
    data_column="image",
    model_input_size=128,
    max_order=2,
    hierarchy=4,
    hips_id="Emoji_Dataset_Projection",
    hips_name="Emoji dataset projection",
    hips_path="dataset",
    root_path=OUTPUT_PATH,
    distortion_correction=False,
)

# Register the survey as a layer too, and re-render the standalone page so it
# offers both the data survey and the real emoji catalogue.
dataset_projection.register(html_generator)
dataset_projection.execute()
html_generator.generate()

print(open(os.path.join(OUTPUT_PATH, "dataset", "properties")).read())

## 5. Serve the tiles

Aladin Lite runs in the browser, so it fetches tiles over HTTP rather than from
the notebook's filesystem. A `SimpleHTTPRequestHandler` on `output/` is enough,
with one addition: the page lives on the Jupyter origin while the tiles come
from port 8083, so the responses need
`Access-Control-Allow-Origin` — WebGL refuses to texture cross-origin images
without it, and the `properties` file is read with `fetch`.

The server runs on a daemon thread, so it goes away with the kernel.
`allow_reuse_address` lets the cell be re-run without waiting for the socket's
`TIME_WAIT` to expire.

In [ ]:
import functools
import http.server
import socketserver
import threading


class CORSRequestHandler(http.server.SimpleHTTPRequestHandler):
    """Static file handler that allows cross-origin reads and stays quiet."""

    def end_headers(self) -> None:
        self.send_header("Access-Control-Allow-Origin", "*")
        super().end_headers()

    def log_message(self, *args) -> None:
        pass


class ReusableThreadingServer(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


# Shut down a server left behind by an earlier run of this cell.
if (previous := globals().get("httpd")) is not None:
    previous.shutdown()
    previous.server_close()

httpd = ReusableThreadingServer(
    ("", PORT),
    functools.partial(CORSRequestHandler, directory=os.path.abspath(OUTPUT_PATH)),
)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"serving {os.path.abspath(OUTPUT_PATH)} on {BASE_URL}")

## 6. Explore it with ipyaladin

`ipyaladin` embeds Aladin Lite as a Jupyter widget. A few calls do the work:

- `survey` — the base image layer. Point it at the HiPS directory and Aladin
  reads `properties` to discover the tile format, size and maximum order.
- `add_catalog_from_URL` — loads the VOTable written above from the same server.
  Reading it by URL keeps the widget's message channel free; the in-memory
  equivalent is `aladin.add_table(catalog, ...)`, which is more convenient for
  small tables but ships every row through the kernel-browser connection.
  Option names are converted to Aladin Lite's camelCase, so `source_size`
  becomes `sourceSize`; `on_click="showTable"` makes a click show the full
  catalogue row (including its `text` label) in a popup and also fills
  `aladin.clicked_object`.

`fov=180` starts zoomed out to the whole sphere. Then: **scroll** to zoom,
**drag** to pan, and click a marker to see which emoji it is. The background
you are flying over is the real dataset projected onto the sphere, and the
markers are the same 2,749 real emoji at the positions the encoder assigned
them — so a marker should sit close to the tile it lands on.

If the widget stays blank: the widget pulls Aladin Lite itself from a CDN, so it
needs internet access, and the tiles must be reachable from the browser — check
that the server cell above is still running, and if Jupyter runs in the
container from `compose.yml`, that port 8083 is published.

In [ ]:
from ipyaladin import Aladin

WIDGET_HEIGHT = 800

aladin = Aladin(
    survey=f"{BASE_URL}/dataset",  # the HiPS tiling generated above
    target="0 +0",
    fov=180,  # degrees: start with the whole sphere in view
    height=WIDGET_HEIGHT,
)

# Work around https://github.com/jupyterlab/jupyterlab/issues/16630: notebook
# front ends (JupyterLab and VS Code alike) virtualize cell output that is
# scrolled out of view, and Aladin Lite reports that transient collapse as a
# resize. ipyaladin bakes whatever it reports back into `_height`, so the
# widget loses a bit of height every time it leaves and re-enters view,
# eventually shrinking to 1px. Snapping `_height` straight back stops that
# ratchet instead of letting each round-trip shrink it further.
aladin.observe(
    lambda change: (
        setattr(aladin, "_height", WIDGET_HEIGHT)
        if change["new"] != WIDGET_HEIGHT
        else None
    ),
    names="_height",
)

aladin

In [ ]:
# Overlay the real emoji. Aladin picks up 'ra'/'dec' from the UCDs, but naming
# the fields explicitly is robust against differently named columns.
_ = aladin.add_catalog_from_URL(
    f"{BASE_URL}/catalog.vot",
    {
        "name": "Emoji dataset",
        "color": "#ff3b30",
        "shape": "circle",
        "source_size": 8,
        "on_click": "showTable",
        "ra_field": "ra",
        "dec_field": "dec",
    },
)

In [ ]:
# Click a marker in the widget above, then re-run this cell.
aladin.clicked_object

## Where to go next

`output/index.html` is a standalone page generated by `HTMLGenerator` — open
<http://localhost:8083/index.html> for the same view outside the notebook, ready
to be published to any static web host.

Some directions worth trying:

- **Train your own model.** [`emojis_training.ipynb`](emojis_training.ipynb)
  trains a small demo model from scratch and exports it to
  `output/onnx/encoder.onnx`. Point `encoder_path` above at that file instead
  of the Hugging Face download to explore it here — just change
  `resize_image`'s target size back to 64 × 64 to match, since the demo model
  was trained at that resolution.
- **Sharper sky.** Raise `max_order` in `DatasetProjection`; each order
  quadruples the tile count.
- **Other datasets.** Nothing below the model download is Emoji-specific;
  point `HF_REPO`/`HF_SUBDIR` at a different Spherinator model on Hugging Face
  and `catalog_datamodule`/`DATASET_REPO` at its matching dataset.

Further reading: Polsterer, Doser, Fehlner & Trujillo-Gomez,
[*Spherinator and HiPSter: Representation Learning for Unbiased Knowledge
Discovery from Simulations*](https://arxiv.org/abs/2406.03810) (2024), and the
[Spherinator documentation](https://spherinator.readthedocs.io).